In [90]:
import data_generation_utils as dgu
import random

instances = dgu.read_instances("instances.txt")
instances

[Instance(n_processors=2, tasks=[Task(id=0, r=4, l=8, w=5), Task(id=1, r=6, l=9, w=11), Task(id=2, r=5, l=10, w=10), Task(id=3, r=3, l=9, w=1), Task(id=4, r=4, l=10, w=3), Task(id=5, r=3, l=5, w=9), Task(id=6, r=5, l=1, w=9)])]

In [91]:
def objective(schedule):
    total = 0

    for processor in schedule:
        current_time = 0

        for task in processor:
            start = max(current_time, task.r)
            completion = start + task.l

            total += task.w * completion
            current_time = completion

    return total

In [92]:
def local_search(instance, initial_solution):
    current = [processor_load.copy() for processor_load in initial_solution]
    current_value = objective(current)

    improved = True

    # Nearest neighborhood: 
    # looping over swaps of 1 pair of tasks and relocations of 1 task
    
    while improved:
        improved = False

        # try all swaps
        for p1 in range(len(current)):
            for p2 in range(p1, len(current)):

                for i in range(len(current[p1])):
                    for j in range(len(current[p2])):

                        if p1 == p2 and i == j:
                            continue

                        # swap
                        current[p1][i], current[p2][j] = (
                            current[p2][j],
                            current[p1][i]
                        )

                        new_value = objective(current)

                        if new_value < current_value:
                            current_value = new_value
                            improved = True
                            break

                        # undo
                        current[p1][i], current[p2][j] = (
                            current[p2][j],
                            current[p1][i]
                        )

                    if improved:
                        break

                if improved:
                    break

            if improved:
                break

        if improved:
            continue

#         try all relocations:
        for p1 in range(len(current)):
            for i in range(len(current[p1])):

                task = current[p1][i]

                for p2 in range(len(current)):
                    for j in range(len(current[p2]) + 1):

                        if p1 == p2:
                            continue

                        # Remove task
                        current[p1].pop(i)

                        # Insert into another processor
                        current[p2].insert(j, task)

                        new_value = objective(current)

                        if new_value < current_value:
                            current_value = new_value
                            improved = True
                            break

                        # Undo
                        current[p2].pop(j)
                        current[p1].insert(i, task)

                    if improved:
                        break

                if improved:
                    break

            if improved:
                break

    return current

In [93]:
from typing import List
from data_generation_utils import Task, Instance
Chromosome = List[int]
Solution = List[List[Task]]

def decode(instance: Instance, chromosome: Chromosome) -> Solution:
    tasks_by_id = {t.id: t for t in instance.tasks}
    free_time = [0] * instance.n_processors
    solution: Solution = [[] for _ in range(instance.n_processors)]
    
    for task_id in chromosome:
        task = tasks_by_id[task_id]
        processor = min(range(instance.n_processors), key=lambda p:max(free_time[p], task.r))
        start = max(free_time[processor], task.r)
        free_time[processor] = start + task.l
        solution[processor].append(task)
        
    return solution

example_chromosome = range(len(instances[0].tasks))
example_solution = decode(instances[0], example_chromosome)
example_solution

[[Task(id=0, r=4, l=8, w=5),
  Task(id=2, r=5, l=10, w=10),
  Task(id=4, r=4, l=10, w=3)],
 [Task(id=1, r=6, l=9, w=11),
  Task(id=3, r=3, l=9, w=1),
  Task(id=5, r=3, l=5, w=9),
  Task(id=6, r=5, l=1, w=9)]]

In [94]:
def encode(solution: Solution) -> Chromosome:

    events = []  # (start_time, task_id)
    for processor_schedule in solution:
        current_time = 0
        for task in processor_schedule:
            start = max(current_time, task.r)
            completion = start + task.l
            events.append((start, task.id))
            current_time = completion

    # sort chronologically; task.id as tiebreaker keeps it deterministic
    events.sort(key=lambda e: (e[0], e[1]))

    return [task_id for _, task_id in events]

In [95]:
def random_chromosome(instance: Instance) -> Chromosome:
    ids = [t.id for t in instance.tasks]
    random.shuffle(ids)
    return ids

In [96]:
def crossover(parent1:Chromosome, parent2: Chromosome) -> Chromosome:
    
    n = len(parent1)
    a, b = sorted(random.sample(range(n), 2))
    
    child: List[int] = [None] * n
    child[a:b] = parent1[a:b]
    
    taken = set(child[a:b])
    filler = (gene for gene in parent2 if gene not in taken)
    for i in range(n):
        if child[i] == None:
            child[i] = next(filler)

    return child

In [97]:
def swap_mutation(chromosome: Chromosome) -> Chromosome:
    mutated = chromosome.copy()
    i, j = random.sample(range(len(mutated)), 2)
    mutated[i], mutated[j] = mutated[j], mutated[i]
    return mutated

def tournament_selection(
    population: List[Chromosome], costs: List[int], k: int = 3
) -> Chromosome:
    contenders = random.sample(range(len(population)), k)
    best = min(contenders, key=lambda i: costs[i])
    return population[best]

In [98]:
def chromosome_cost(instance: Instance, chromosome: Chromosome) -> int:
    return objective(decode(instance, chromosome))

In [99]:
def memetic_algorithm(
    instance: Instance,
    pop_size: int = 100,
    generations: int = 100,
    crossover_rate: float = 0.9,
    mutation_rate: float = 0.2,
    tournament_size: int = 4,
    elitism: int = 2,
    ls_elitism: int = 2,
    local_search_rate: float = 0.1,
    verbose: bool = True,
) -> Solution:

    population = [random_chromosome(instance) for _ in range(pop_size)]

    best_chromosome = None
    best_cost = float("inf")

    for gen in range(generations):
        costs = [chromosome_cost(instance, c) for c in population]
        
        # memetic step: local-search refine some individuals
        ranked = sorted(range(len(population)), key=lambda i: costs[i])
        refine_idxs = set(ranked[:ls_elitism])
        for i in range(len(population)):
            if i not in refine_idxs and random.random() < local_search_rate:
                refine_idxs.add(i)
        
        for idx in refine_idxs:
            solution = decode(instance, population[idx])
            improved_solution = local_search(instance, solution)
            improved_cost = objective(improved_solution)
            if improved_cost < costs[idx]:
                population[idx] = encode(improved_solution)
                costs[idx] = improved_cost
        
        # track best
        gen_best_idx = costs.index(min(costs))
        if costs[gen_best_idx] < best_cost:
            best_cost = costs[gen_best_idx]
            best_chromosome = population[gen_best_idx].copy()
    # next generation: elitism + crossover/mutation
        ranked = sorted(range(len(population)), key=lambda i: costs[i])
        new_population = [population[i].copy() for i in ranked[:elitism]]


        while len(new_population) < pop_size:
            parent1 = tournament_selection(population, costs, tournament_size)
            parent2 = tournament_selection(population, costs, tournament_size)

            child = crossover(parent1, parent2) if random.random() < crossover_rate else parent1.copy()
            if random.random() < mutation_rate:
                child = swap_mutation(child)

            new_population.append(child)

        population = new_population

        if verbose:
            print(f"\rgen {gen:4d} | best cost so far: {best_cost}", end="")

    best_solution = decode(instance, best_chromosome)

    if verbose:
        print(f"\rMemetic done. Best objective: {objective(best_solution)}", end="")

    return best_solution



In [100]:
memetic_algorithm(instances[0])

Memetic done. Best objective: 702

[[Task(id=5, r=3, l=5, w=9),
  Task(id=2, r=5, l=10, w=10),
  Task(id=4, r=4, l=10, w=3)],
 [Task(id=6, r=5, l=1, w=9),
  Task(id=1, r=6, l=9, w=11),
  Task(id=0, r=4, l=8, w=5),
  Task(id=3, r=3, l=9, w=1)]]

In [101]:
import csv
import time

header = ["dimension", "average_objective", "time"]

dimensions = range(4, 26)

instance_sets = {
    k: dgu.read_instances(f"data/{k}_tasks.txt")
    for k in dimensions
}

with open("reports/memetic_algorithm.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)

    for dimension in dimensions:
        instances = instance_sets[dimension]

        start = time.perf_counter()
        solutions = [memetic_algorithm(instance) for instance in instances]
        duration = time.perf_counter() - start

        objectives = [objective(solution) for solution in solutions]
        average_objective = sum(objectives) / len(objectives)

        writer.writerow([
            dimension,
            average_objective,
            duration
        ])

Memetic done. Best objective: 97286